# BMEN 600 pipeline: CSF biomarkers vs hippocampal subfield volumes

Runs the whole analysis in order. Put the ADNI downloads and `ADNI_Participants.xlsx` in `data/raw/` first
(see `data/README.md`).

1. `code/01_select_and_match.py` - how the AD participants were matched to controls (already done; rerun to reproduce)
2. `code/02_preprocessing.ipynb` - scans → HippUnfold → subfield volumes, eTIV, QC sheet (**run that notebook separately**;
   HippUnfold itself runs in a terminal)
3. `code/03_merge.py` - one row per participant + Table 1
4. `code/04_analysis.py` - statistics, tables and the heatmap

**Before committing this notebook:** only aggregate results are displayed, but clear all outputs if you add cells that
print participant-level data. The repo is public and ADNI data cannot be shared.

In [ ]:
# Setup: find the BMEN-600_code folder (the one containing code/config.py) and work from there
import os, sys
from pathlib import Path
root = Path.cwd()
while not (root / "code" / "config.py").exists():
    if root.parent == root:
        raise FileNotFoundError("Open this notebook from inside the BMEN-600_code folder.")
    root = root.parent
os.chdir(root); sys.path.insert(0, str(root / "code"))
import config as cfg
needed = [cfg.PARTICIPANTS_FILE, cfg.CSF_FILE, *cfg.IDA_EXPORTS.values()]
print("Working in", root)
print("Missing raw files:", [p.name for p in needed if not p.exists()] or "none")

## Step 1: matching (reproduces the AD-control matching)

In [ ]:
%run code/01_select_and_match.py

## Step 2: preprocessing

Open and run `code/02_preprocessing.ipynb` (and HippUnfold) before continuing. It writes `data/derived/volumes.csv`
(subfield volumes + eTIV) and `data/derived/qc.csv`.

In [ ]:
## Step 3: merge + Table 1 (runs without volumes too: then only Table 1 and CSF counts)
from IPython.display import Markdown, display
%run code/03_merge.py
display(Markdown(Path("outputs/table1_demographics.md").read_text()))

In [ ]:
## Step 4: analysis
import pandas as pd
from IPython.display import Image
%run code/04_analysis.py
heat = Path("outputs/figures/partial_correlation_heatmap.png")
if heat.exists():
    display(Image(str(heat)))
    corr = pd.read_csv("outputs/results/partial_correlations.csv")
    print("FDR-significant primary correlations:")
    display(corr[(corr.analysis == "primary") & corr.significant_fdr].round(3))
    print("Does the association differ by group? (biomarker x group interaction)")
    display(pd.read_csv("outputs/results/interaction_marker_x_group.csv").round(3))